# Day 2: SQL Analysis
This notebook runs analytical queries on the `bluestock_mf.db` to extract insights.

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

# Connect to Database
db_path = r'../../bluestock_mf.db'
engine = create_engine(f'sqlite:///{db_path}')

def run_query(query):
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn)

## 1. Top 5 Funds by AUM

In [ ]:
query1 = """
SELECT scheme_name, aum_crore 
FROM fact_performance p
JOIN dim_fund f ON p.amfi_code = f.amfi_code
ORDER BY aum_crore DESC 
LIMIT 5;
"""
run_query(query1)

## 2. SIP Year-over-Year (YoY) Growth

In [ ]:
query2 = """
WITH YearlySIP AS (
    SELECT d.year, SUM(amount_inr) as total_sip
    FROM fact_transactions t
    JOIN dim_date d ON t.transaction_date = d.date
    WHERE t.transaction_type = 'SIP'
    GROUP BY d.year
)
SELECT curr.year, curr.total_sip, 
       ((curr.total_sip - prev.total_sip) * 100.0 / prev.total_sip) as yoy_growth_pct
FROM YearlySIP curr
LEFT JOIN YearlySIP prev ON curr.year = prev.year + 1;
"""
run_query(query2)

## 3. Transaction Amount by State

In [ ]:
query3 = """
SELECT state, SUM(amount_inr) as total_amount
FROM fact_transactions
GROUP BY state
ORDER BY total_amount DESC
LIMIT 10;
"""
run_query(query3)

## 4. High Rated Funds with 3-Year Returns

In [ ]:
query4 = """
SELECT scheme_name, return_3yr_pct, morningstar_rating
FROM fact_performance p
JOIN dim_fund f ON p.amfi_code = f.amfi_code
WHERE morningstar_rating >= 4
ORDER BY return_3yr_pct DESC
LIMIT 10;
"""
run_query(query4)